# Práctica de Data Cleaning


In [1]:
import pandas as pd

## Objetivo

El objetivo de este ejercicio es aplicar técnicas de **limpieza de datos** en Pandas.

Este ejercicio cubre la manipulación de strings (utilizando métodos de strings), el uso de expresiones regulares, la importancia de la limpieza y la conversión a `datetime`

## Contexto del Conjunto de Datos

Hemos cargado un DataFrame que contiene información de Presidentes de EE. UU. La columna `'Born'` (Nacimiento) contiene la fecha y, entre paréntesis, la ciudad o ubicación, como por ejemplo: `February 22, 1732[b] (Westmoreland County, Virginia)`.

## Instrucciones

- De la columna `'President'` Crea una columna para el nombre 'firstname' y otra para el apellido (lastname) de los presidentes eliminando la referencias a las notas al pie que encuentres. Puedes utilizar un patron de Regular Expression para realizar esta tarea
- La columna `'Born'` contiene referencias entre corchetes (ej. `[b]`) y texto adicional entre paréntesis (ej. `(Westmoreland County, Virginia)`). Debemos eliminar todo este texto para quedarnos solo con la fecha limpia.
- Convierte la columna `'Born'` al Tipo `datetime`
- Extrae de la columna  `'Born'` el año de nacimiento y genera una nueva columna denominada `'Year Born'`


In [2]:
df = pd.read_csv("../data/presidents.csv")

df.head()

,#,President,Born,Age atstart of presidency,Age atend of presidency,Post-presidencytimespan,Died,Age
0,1,George Washington,"Feb 22, 1732[a]","57 years, 67 daysApr 30, 1789","65 years, 10 daysMar 4, 1797","2 years, 285 days","Dec 14, 1799","67 years, 295 days"
1,2,John Adams,"Oct 30, 1735[a]","61 years, 125 daysMar 4, 1797","65 years, 125 daysMar 4, 1801","25 years, 122 days","Jul 4, 1826","90 years, 247 days"
2,3,Thomas Jefferson,"Apr 13, 1743[a]","57 years, 325 daysMar 4, 1801","65 years, 325 daysMar 4, 1809","17 years, 122 days","Jul 4, 1826","83 years, 82 days"
3,4,James Madison,"Mar 16, 1751[a]","57 years, 353 daysMar 4, 1809","65 years, 353 daysMar 4, 1817","19 years, 116 days","Jun 28, 1836","85 years, 104 days"
4,5,James Monroe,"Apr 28, 1758","58 years, 310 daysMar 4, 1817","66 years, 310 daysMar 4, 1825","6 years, 122 days","Jul 4, 1831","73 years, 67 days"


## 1. Inspección inicial

Comprobamos las columnas, los tipos y los valores faltantes antes de modificar el DataFrame. Los datos son una instantánea histórica del archivo proporcionado; no se actualizan a la actualidad.


In [3]:
filas_originales = len(df)
columnas_originales = df.columns.tolist()
df_original = df.copy(deep=True)
df.info()
df.isna().sum()


<class 'pandas.DataFrame'>
RangeIndex: 44 entries, 0 to 43
Data columns (total 8 columns):
 #   Column                     Non-Null Count  Dtype
---  ------                     --------------  -----
 0   #                          44 non-null     int64
 1   President                  44 non-null     str  
 2   Born                       44 non-null     str  
 3   Age atstart of presidency  44 non-null     str  
 4   Age atend of presidency    44 non-null     str  
 5   Post-presidencytimespan    36 non-null     str  
 6   Died                       44 non-null     str  
 7   Age                        44 non-null     str  
dtypes: int64(1), str(7)
memory usage: 2.9 KB


#                            0
President                    0
Born                         0
Age atstart of presidency    0
Age atend of presidency      0
Post-presidencytimespan      8
Died                         0
Age                          0
dtype: int64

## 2. Separar nombre y apellido

Eliminamos referencias entre corchetes y normalizamos los espacios. Usamos el primer término como `firstname` y el último como `lastname`, sin incluir segundos nombres o iniciales. En este conjunto, el apellido compuesto `Van Buren` se conserva completo. Esta convención está adaptada a los nombres del CSV; otros apellidos compuestos requerirían revisar la regla.


In [4]:
nombres_limpios = (
    df['President'].str.replace(r'\[[^\]]*\]', '', regex=True)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

df['firstname'] = nombres_limpios.str.extract(r'^(\S+)', expand=False)
df['lastname'] = nombres_limpios.str.extract(r'((?:Van\s+)?\S+)$', expand=False)

df[['President', 'firstname', 'lastname']]


,President,firstname,lastname
0,George Washington,George,Washington
1,John Adams,John,Adams
2,Thomas Jefferson,Thomas,Jefferson
3,James Madison,James,Madison
4,James Monroe,James,Monroe
5,John Quincy Adams,John,Adams
6,Andrew Jackson,Andrew,Jackson
7,Martin Van Buren,Martin,Van Buren
8,William H. Harrison,William,Harrison
9,John Tyler,John,Tyler


## 3. Limpiar la columna Born

Quitamos las notas entre corchetes y las ubicaciones entre paréntesis. Finalmente, normalizamos los espacios. Aunque este CSV no contiene ubicaciones en `Born`, la expresión también cubre el caso descrito en la consigna.


In [5]:
def limpiar_fecha(serie):
    return (
        serie.str.replace(r'\[[^\]]*\]', '', regex=True)
        .str.replace(r'\([^)]*\)', '', regex=True)
        .str.replace(r'\s+', ' ', regex=True)
        .str.strip()
    )

df['Born'] = limpiar_fecha(df['Born'])
df[['President', 'Born']].head()


,President,Born
0,George Washington,"Feb 22, 1732"
1,John Adams,"Oct 30, 1735"
2,Thomas Jefferson,"Apr 13, 1743"
3,James Madison,"Mar 16, 1751"
4,James Monroe,"Apr 28, 1758"


## 4. Convertir Born a datetime y extraer el año

El archivo usa meses abreviados en inglés, por ejemplo `Feb 22, 1732`, por eso indicamos el formato `%b %d, %Y`. Usamos `errors='raise'` para detectar fechas inválidas sin convertirlas silenciosamente en valores faltantes. `.dt.year` obtiene el año una vez convertida la columna.


In [6]:
df['Born'] = pd.to_datetime(df['Born'], format='%b %d, %Y', errors='raise')
df['Year Born'] = df['Born'].dt.year

df[['President', 'firstname', 'lastname', 'Born', 'Year Born']]


,President,firstname,lastname,Born,Year Born
0,George Washington,George,Washington,1732-02-22,1732
1,John Adams,John,Adams,1735-10-30,1735
2,Thomas Jefferson,Thomas,Jefferson,1743-04-13,1743
3,James Madison,James,Madison,1751-03-16,1751
4,James Monroe,James,Monroe,1758-04-28,1758
5,John Quincy Adams,John,Adams,1767-07-11,1767
6,Andrew Jackson,Andrew,Jackson,1767-03-15,1767
7,Martin Van Buren,Martin,Van Buren,1782-12-05,1782
8,William H. Harrison,William,Harrison,1773-02-09,1773
9,John Tyler,John,Tyler,1790-03-29,1790


## 5. Verificación

Verificamos la conservación de filas y columnas originales, las fechas convertidas, los años y algunos nombres representativos. No eliminamos filas por los nulos de otras columnas: su tratamiento no forma parte de esta consigna. Tampoco eliminamos las dos presidencias de Grover Cleveland, que son registros distintos en la fuente.


In [7]:
assert len(df) == filas_originales == 44
assert df.index.equals(df_original.index)
assert set(columnas_originales).issubset(df.columns)
assert pd.api.types.is_datetime64_any_dtype(df['Born'])
assert df[['firstname', 'lastname', 'Born', 'Year Born']].notna().all().all()
assert df['Year Born'].equals(df['Born'].dt.year)
assert not df[['firstname', 'lastname']].apply(
    lambda columna: columna.str.contains(r'\[|\]', regex=True)
).any().any()

washington = df.loc[df['President'].eq('George Washington')].iloc[0]
assert washington['firstname'] == 'George'
assert washington['lastname'] == 'Washington'
assert washington['Born'] == pd.Timestamp('1732-02-22')
assert washington['Year Born'] == 1732
assert df.loc[df['President'].eq('John Quincy Adams'), 'lastname'].iloc[0] == 'Adams'
assert df.loc[df['President'].eq('Martin Van Buren'), 'lastname'].iloc[0] == 'Van Buren'

# Comprobamos también el ejemplo con ubicación y nota al pie del enunciado.
ejemplo = pd.Series(['February 22, 1732[b] (Westmoreland County, Virginia)'])
assert limpiar_fecha(ejemplo).iloc[0] == 'February 22, 1732'
assert pd.to_datetime(limpiar_fecha(ejemplo), format='%B %d, %Y').iloc[0] == pd.Timestamp('1732-02-22')

# Las columnas ajenas a la consigna conservan sus valores originales.
sin_cambios = [col for col in columnas_originales if col != 'Born']
pd.testing.assert_frame_equal(df[sin_cambios], df_original[sin_cambios])
print(f'Verificación correcta: {len(df)} filas, fechas sin nulos y las cuatro consignas resueltas.')
df[['firstname', 'lastname', 'Born', 'Year Born']].dtypes


Verificación correcta: 44 filas, fechas sin nulos y las cuatro consignas resueltas.


firstname               str
lastname                str
Born         datetime64[us]
Year Born             int32
dtype: object